# Módulo 11 — Evaluación de Riesgos y Comportamiento del Cliente
### Diagnóstico y Predictibilidad (92-0030) · Profesor Robin Sequeira

En este notebook vamos a hacer tres cosas, en el mismo orden que las vimos en clase:

1. **Reconstruir el modelo de churn** con el dataset Telco (igual que en el Caso I, pero más rápido).
2. **Leer la curva ROC, el AUC y el punto de corte óptimo**, para entender el riesgo como una probabilidad y no solo como un "sí" o "no".
3. **Segmentar a los clientes con K-Means** usando la probabilidad de churn, y describir cada segmento para proponer estrategias de retención.

> **Nota importante:** el dataset ya viene **embebido directamente en el código** (en la siguiente celda). No es necesario descargar ni subir ningún archivo a Databricks: solo hay que ejecutar las celdas en orden, de arriba hacia abajo.


## Parte 0 · Preparar el ambiente y cargar los datos

Primero importamos las librerías que vamos a necesitar. Todas vienen preinstaladas en Databricks Community Edition, así que no hace falta instalar nada.


In [ ]:
# Librerías para manejar datos
import pandas as pd
import numpy as np
import io

# Librerías para graficar
import matplotlib.pyplot as plt

# Librerías de scikit-learn para el modelo, la evaluación y el clustering
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_curve, auc
from sklearn.cluster import KMeans

print("Librerías cargadas correctamente. Listos para empezar.")


### Dataset embebido: Telco Customer Churn (versión de clase)

Este es un dataset con la misma estructura y las mismas variables que el Telco Customer Churn que usamos en el Caso I (7,043 clientes, 21 variables), pero preparado especialmente para esta clase con 400 clientes, para que las celdas corran rápido durante la sesión.

**¿Por qué está embebido como texto (CSV) dentro del código?**
Porque así nadie tiene que subir archivos a Databricks ni preocuparse por rutas de carpetas: el dato "vive" dentro del notebook mismo. Usamos `io.StringIO` para que pandas pueda leer ese texto exactamente igual que si fuera un archivo `.csv`.


In [ ]:
# El dataset completo va como texto, dentro de una variable de Python.
# io.StringIO() hace que pandas "vea" este texto como si fuera un archivo CSV real.
datos_csv = """customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
7000-SDXKU,Male,0,Yes,Yes,38,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Credit card (automatic),65.41,2477.88,Yes
7001-RMQEB,Female,1,No,No,47,Yes,No,DSL,No,Yes,Yes,No,No,No,Month-to-month,Yes,Mailed check,50.47,2342.71,No
7002-UZWHJ,Female,0,No,Yes,56,Yes,Yes,DSL,No,No,No,Yes,No,No,Month-to-month,Yes,Mailed check,51.16,2711.25,No
7003-JSQVY,Male,0,No,Yes,18,Yes,No,Fiber optic,No,No,Yes,No,No,Yes,Month-to-month,Yes,Credit card (automatic),79.88,1391.67,Yes
7004-KRNYJ,Male,0,No,No,1,Yes,Yes,Fiber optic,Yes,No,No,No,No,No,Two year,No,Electronic check,78.76,75.1,No
7005-EKQYV,Female,0,No,No,50,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Credit card (automatic),23.01,1085.49,No
7006-ANDJY,Male,0,No,No,7,No,No phone service,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Bank transfer (automatic),24.29,156.6,Yes
7007-ZDDSK,Female,0,Yes,No,5,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Credit card (automatic),22.43,106.99,No
7008-LWYXU,Male,0,No,No,48,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Credit card (automatic),20.84,945.91,No
7009-ZTXNB,Male,0,Yes,No,58,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Credit card (automatic),24.4,1382.92,No
7010-VYHLP,Female,0,No,No,18,Yes,No,Fiber optic,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,78.18,1303.43,Yes
7011-RSKWV,Female,0,No,No,14,Yes,Yes,DSL,No,No,No,No,No,Yes,Month-to-month,No,Credit card (automatic),51.0,676.2,Yes
7012-YPBWB,Female,0,No,No,30,Yes,No,DSL,No,Yes,No,No,No,No,One year,Yes,Credit card (automatic),54.58,1513.74,No
7013-EGETL,Female,0,No,No,24,Yes,Yes,Fiber optic,No,No,No,No,No,Yes,Month-to-month,No,Bank transfer (automatic),65.12,1538.1,No
7014-FAFSV,Female,0,Yes,No,58,Yes,Yes,DSL,No,Yes,No,No,No,Yes,Month-to-month,Yes,Electronic check,52.84,2993.68,No
7015-HBPUB,Female,0,No,Yes,7,Yes,No,DSL,No,Yes,No,No,No,Yes,Month-to-month,Yes,Credit card (automatic),52.78,352.37,Yes
7016-VEGQB,Female,0,No,No,10,No,No phone service,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,No,Electronic check,24.61,230.04,No
7017-HDFUF,Male,0,Yes,No,9,Yes,No,DSL,No,No,No,No,Yes,No,Month-to-month,Yes,Mailed check,55.79,510.04,Yes
7018-BFCQZ,Female,1,Yes,Yes,7,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Mailed check,18.25,125.0,No
7019-ETJBS,Male,0,Yes,No,4,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,No,Electronic check,75.64,298.9,Yes
7020-FNNQD,Female,0,No,No,58,Yes,No,Fiber optic,No,No,Yes,Yes,No,No,One year,Yes,Bank transfer (automatic),74.31,4194.19,No
7021-GTYAK,Male,0,No,No,22,Yes,No,Fiber optic,No,No,Yes,No,No,Yes,Month-to-month,No,Electronic check,74.44,1638.75,No
7022-XGLXZ,Male,0,No,No,11,Yes,No,DSL,No,No,No,Yes,Yes,Yes,Month-to-month,No,Bank transfer (automatic),57.56,639.26,Yes
7023-RSQGX,Female,1,Yes,No,11,Yes,Yes,Fiber optic,No,No,No,No,Yes,No,Month-to-month,No,Credit card (automatic),73.23,771.92,No
7024-ABQWV,Female,0,No,No,4,Yes,Yes,DSL,No,No,No,Yes,No,No,Two year,Yes,Mailed check,43.27,171.18,No
7025-SKJLG,Female,1,Yes,No,56,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Bank transfer (automatic),22.6,1193.45,No
7026-LCHYS,Male,0,No,No,61,No,No phone service,DSL,No,No,No,No,Yes,Yes,Month-to-month,No,Electronic check,56.72,3408.66,Yes
7027-DGGGS,Female,0,Yes,No,38,Yes,Yes,DSL,No,No,No,Yes,No,No,Month-to-month,Yes,Credit card (automatic),53.72,1959.53,Yes
7028-XTMDD,Female,0,No,No,63,Yes,Yes,Fiber optic,No,No,Yes,No,No,No,Two year,Yes,Credit card (automatic),72.77,4220.86,No
7029-VSNPW,Male,0,No,No,52,Yes,Yes,Fiber optic,No,No,No,No,No,Yes,Month-to-month,Yes,Electronic check,74.34,3925.8,No
7030-VCKBK,Male,0,No,No,7,Yes,No,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),83.38,587.65,Yes
7031-RFDMN,Male,0,No,No,9,Yes,Yes,Fiber optic,No,Yes,No,Yes,No,No,Month-to-month,No,Mailed check,85.75,732.46,Yes
7032-TAFUS,Male,1,Yes,No,4,Yes,No,DSL,No,No,No,No,No,Yes,One year,No,Mailed check,38.16,155.66,No
7033-XVWNZ,Female,0,Yes,No,11,Yes,No,Fiber optic,Yes,No,No,No,No,Yes,One year,No,Bank transfer (automatic),75.17,786.94,No
7034-KXKLS,Female,1,Yes,No,72,Yes,Yes,DSL,No,No,No,No,Yes,Yes,One year,No,Mailed check,54.02,3756.75,No
7035-KDXLY,Male,0,Yes,No,54,No,No phone service,DSL,No,No,No,Yes,No,No,Two year,Yes,Bank transfer (automatic),45.16,2451.14,No
7036-KZQUU,Female,0,No,No,28,Yes,No,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,48.47,1289.93,Yes
7037-EVYRY,Female,0,No,No,64,Yes,No,DSL,No,No,No,No,Yes,Yes,Month-to-month,Yes,Mailed check,53.5,3304.34,No
7038-WRMWE,Male,0,No,No,15,Yes,Yes,DSL,Yes,Yes,No,Yes,No,No,Month-to-month,No,Credit card (automatic),55.87,826.79,Yes
7039-PVYGC,Female,0,No,Yes,7,Yes,Yes,Fiber optic,No,No,Yes,No,No,No,Month-to-month,Yes,Electronic check,74.76,488.28,Yes
7040-JWNBC,Male,0,No,Yes,32,Yes,Yes,DSL,No,Yes,Yes,Yes,No,Yes,Month-to-month,Yes,Bank transfer (automatic),65.51,2017.15,Yes
7041-JJDMM,Female,0,No,No,5,No,No phone service,DSL,No,No,No,No,Yes,No,Month-to-month,Yes,Electronic check,48.55,230.84,Yes
7042-EFRYZ,Female,0,Yes,No,59,Yes,No,Fiber optic,Yes,No,Yes,No,No,Yes,Month-to-month,No,Credit card (automatic),81.69,4760.32,No
7043-PAXYG,Male,0,Yes,No,21,Yes,Yes,DSL,Yes,No,No,Yes,Yes,Yes,One year,No,Bank transfer (automatic),62.34,1222.21,No
7044-KAVUA,Male,0,No,Yes,70,Yes,Yes,Fiber optic,No,Yes,No,No,No,No,Month-to-month,No,Mailed check,74.47,4917.19,No
7045-SKPFV,Female,1,No,No,66,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Electronic check,18.25,1193.83,No
7046-KWDQH,Male,0,Yes,Yes,17,Yes,Yes,Fiber optic,Yes,No,Yes,No,Yes,Yes,One year,No,Mailed check,89.85,1429.12,No
7047-ZPTCS,Female,0,No,No,71,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Credit card (automatic),18.25, ,No
7048-CJFAQ,Female,0,No,No,15,Yes,No,DSL,No,No,Yes,No,Yes,No,One year,Yes,Mailed check,59.05,878.03,No
7049-FNUET,Female,0,Yes,Yes,68,Yes,Yes,DSL,Yes,Yes,No,Yes,No,Yes,Two year,Yes,Mailed check,65.36,4429.13,No
7050-HHPRR,Female,0,Yes,No,60,Yes,Yes,Fiber optic,No,No,No,No,No,No,One year,Yes,Bank transfer (automatic),74.68,4164.85,No
7051-EPGGY,Male,1,No,Yes,0,Yes,No,Fiber optic,No,Yes,Yes,No,No,No,Month-to-month,Yes,Mailed check,74.96,0.0,Yes
7052-CKLFU,Male,0,No,Yes,34,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Mailed check,18.51,624.56,No
7053-MTKXT,Male,0,Yes,Yes,6,Yes,No,DSL,No,No,Yes,Yes,Yes,No,One year,Yes,Electronic check,53.53,306.38,No
7054-HZWQA,Male,0,No,Yes,54,Yes,No,Fiber optic,Yes,No,No,Yes,Yes,Yes,Month-to-month,No,Electronic check,85.18,4583.07,Yes
7055-QSJDP,Male,0,Yes,No,69,Yes,No,Fiber optic,No,Yes,No,No,No,Yes,Month-to-month,No,Bank transfer (automatic),84.01,5755.13,No
7056-VHLAS,Female,0,No,No,27,Yes,Yes,DSL,No,Yes,Yes,Yes,No,No,Month-to-month,No,Credit card (automatic),64.45,1705.5,Yes
7057-WULBM,Male,0,Yes,Yes,39,Yes,No,DSL,No,Yes,Yes,Yes,No,No,Month-to-month,No,Credit card (automatic),53.91,2051.71,No
7058-YCFWP,Female,0,No,No,29,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Credit card (automatic),18.25,525.79,No
7059-EAKQW,Female,0,Yes,No,2,Yes,No,DSL,Yes,No,Yes,Yes,No,No,Month-to-month,No,Mailed check,58.2,118.72,Yes
7060-XLLYU,Female,0,Yes,No,24,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Electronic check,22.34,524.29,No
7061-FSTFZ,Female,0,No,No,29,Yes,No,Fiber optic,No,No,No,Yes,No,Yes,Two year,Yes,Bank transfer (automatic),71.96,2088.66,No
7062-HPMSX,Male,0,Yes,Yes,42,Yes,Yes,Fiber optic,No,Yes,No,No,No,No,Month-to-month,Yes,Credit card (automatic),68.09,2843.84,No
7063-KKUYX,Female,0,Yes,No,44,Yes,Yes,Fiber optic,No,Yes,No,No,No,No,Two year,No,Bank transfer (automatic),76.46,3383.02,No
7064-USUYA,Male,1,Yes,No,11,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Mailed check,18.25,203.85,Yes
7065-QAPKH,Male,0,No,No,44,Yes,Yes,DSL,No,Yes,No,Yes,Yes,No,Two year,No,Bank transfer (automatic),56.71,2343.8,No
7066-XTPXC,Female,0,No,No,40,Yes,Yes,DSL,No,No,Yes,Yes,No,Yes,Two year,Yes,Bank transfer (automatic),59.15,2215.87,No
7067-RDFEP,Male,0,No,No,37,Yes,No,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Credit card (automatic),85.16,3083.6,No
7068-RULYJ,Male,0,Yes,Yes,57,Yes,No,Fiber optic,No,No,No,No,No,Yes,One year,Yes,Bank transfer (automatic),73.93,4066.22,No
7069-DADLK,Male,0,Yes,No,5,Yes,Yes,DSL,No,No,Yes,No,Yes,Yes,Two year,Yes,Mailed check,57.04,278.66,Yes
7070-WDGZA,Female,0,Yes,No,2,Yes,No,DSL,No,Yes,No,No,Yes,No,Two year,Yes,Electronic check,57.38,114.74,No
7071-LENZF,Male,0,No,No,19,Yes,No,DSL,No,No,No,No,Yes,No,Two year,No,Mailed check,50.64,974.83,No
7072-JLVZC,Male,0,No,No,71,Yes,No,DSL,No,No,No,No,Yes,Yes,One year,Yes,Electronic check,51.41,3510.46,No
7073-MUAVF,Male,0,No,No,47,Yes,No,DSL,No,No,Yes,No,No,Yes,Month-to-month,Yes,Credit card (automatic),54.15,2524.16,Yes
7074-NZRZQ,Female,0,No,No,69,No,No phone service,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,21.53,1395.96,No
7075-FSQHB,Male,0,Yes,No,5,Yes,No,DSL,No,No,No,No,Yes,Yes,One year,Yes,Mailed check,60.87,296.27,No
7076-RPUDN,Male,0,No,No,51,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,23.15,1088.77,No
7077-VXYFE,Male,0,No,No,60,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Bank transfer (automatic),24.86,1447.41,No
7078-PMFCD,Female,1,Yes,No,34,Yes,Yes,DSL,No,No,No,No,Yes,Yes,One year,Yes,Mailed check,51.56, ,No
7079-JTBCG,Female,1,Yes,No,10,Yes,Yes,Fiber optic,Yes,Yes,No,No,No,Yes,Month-to-month,Yes,Electronic check,81.4,808.75,Yes
7080-WZLBA,Female,0,Yes,Yes,10,Yes,Yes,Fiber optic,No,Yes,No,No,No,No,One year,No,Credit card (automatic),73.56,724.53,No
7081-WLBWN,Male,0,Yes,No,49,Yes,No,Fiber optic,No,Yes,Yes,Yes,No,Yes,Month-to-month,Yes,Mailed check,87.43,4094.88,Yes
7082-JLFYU,Male,0,Yes,No,36,Yes,No,DSL,No,No,No,No,Yes,No,One year,No,Mailed check,45.47,1506.9,No
7083-ETELT,Female,0,Yes,No,20,Yes,No,DSL,No,Yes,No,No,No,Yes,Two year,Yes,Electronic check,59.06,1100.35,No
7084-ATFSM,Female,0,No,No,67,Yes,No,DSL,No,No,Yes,Yes,Yes,Yes,Month-to-month,No,Mailed check,66.53,4534.65,No
7085-AUGYW,Female,0,Yes,No,31,Yes,No,Fiber optic,No,Yes,No,No,Yes,No,Month-to-month,No,Bank transfer (automatic),77.78, ,Yes
7086-EQJQP,Male,0,Yes,No,66,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Credit card (automatic),22.57,1418.88,No
7087-HRGXY,Male,0,No,No,16,Yes,No,Fiber optic,Yes,Yes,Yes,No,No,Yes,Two year,Yes,Electronic check,88.89,1345.28,No
7088-DTYDT,Female,0,No,No,3,Yes,No,Fiber optic,No,No,No,Yes,No,No,One year,Yes,Credit card (automatic),77.38,223.7,No
7089-YFPVJ,Female,0,No,No,38,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Bank transfer (automatic),18.25,685.87,No
7090-FKJTJ,Male,0,Yes,Yes,1,Yes,No,Fiber optic,No,Yes,No,Yes,Yes,Yes,Month-to-month,No,Electronic check,87.42,83.55,Yes
7091-KYPYT,Female,0,No,No,67,Yes,Yes,DSL,No,Yes,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,67.27,4176.85,No
7092-XEYYA,Female,1,Yes,Yes,24,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Electronic check,22.49,518.32,No
7093-JKLUS,Male,0,Yes,Yes,1,Yes,No,Fiber optic,Yes,Yes,Yes,Yes,Yes,No,Two year,Yes,Credit card (automatic),89.99,85.14,Yes
7094-FQZLY,Female,1,No,No,39,Yes,Yes,DSL,No,No,No,Yes,No,No,One year,Yes,Electronic check,39.5,1467.32,No
7095-RYXAV,Male,1,No,No,28,Yes,Yes,Fiber optic,Yes,No,Yes,No,No,No,Month-to-month,No,Mailed check,74.67,1957.57,No
7096-EVVGB,Male,1,No,Yes,18,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Bank transfer (automatic),19.76, ,No
7097-LHPBB,Female,0,No,No,37,Yes,Yes,DSL,No,No,Yes,No,No,Yes,Month-to-month,Yes,Credit card (automatic),54.26,1952.74,No
7098-LUHHD,Female,0,No,No,39,Yes,No,Fiber optic,No,No,Yes,No,Yes,Yes,Two year,No,Bank transfer (automatic),90.23,3332.77,No
7099-GHGJG,Male,0,Yes,Yes,22,Yes,Yes,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Bank transfer (automatic),51.83,1112.19,No
7100-SJUQR,Female,0,Yes,Yes,21,Yes,Yes,DSL,No,No,No,No,Yes,No,One year,No,Electronic check,49.83,1014.27,No
7101-QYGZE,Male,0,Yes,No,6,Yes,No,DSL,No,Yes,No,No,No,No,Two year,Yes,Credit card (automatic),49.37,296.1,No
7102-DVXCQ,Female,0,Yes,No,53,No,No phone service,Fiber optic,No,Yes,Yes,No,Yes,No,Month-to-month,Yes,Electronic check,87.92,4732.44,No
7103-NFNSP,Male,0,No,No,22,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Electronic check,18.25,374.63,No
7104-NXFQH,Female,1,Yes,Yes,59,Yes,No,DSL,No,No,No,Yes,No,No,Two year,No,Credit card (automatic),48.94,2911.69,No
7105-ZGWRF,Female,0,No,No,29,Yes,Yes,DSL,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,40.64,1167.34,No
7106-UFGJS,Female,0,No,Yes,58,Yes,No,DSL,No,No,Yes,No,No,No,Month-to-month,Yes,Electronic check,55.18,3166.05,No
7107-KPPEF,Female,0,No,No,5,Yes,Yes,DSL,No,No,No,Yes,No,No,One year,No,Bank transfer (automatic),51.07,244.48,No
7108-HFYKK,Male,0,Yes,No,50,Yes,No,Fiber optic,No,Yes,No,Yes,Yes,No,Month-to-month,No,Electronic check,80.67,3849.54,No
7109-SUSPZ,Female,0,Yes,Yes,24,Yes,No,Fiber optic,Yes,Yes,No,No,No,Yes,Month-to-month,No,Electronic check,83.04,1834.65,Yes
7110-XEDKW,Male,0,No,Yes,71,Yes,No,Fiber optic,Yes,Yes,No,No,No,No,Month-to-month,Yes,Bank transfer (automatic),81.04,5858.68,No
7111-ZSSFQ,Female,0,No,No,0,No,No phone service,Fiber optic,No,Yes,No,Yes,No,No,One year,Yes,Bank transfer (automatic),78.0,0.0,No
7112-ENEBZ,Female,0,No,No,10,Yes,No,Fiber optic,No,Yes,Yes,Yes,No,No,Month-to-month,Yes,Credit card (automatic),81.36,781.25,No
7113-XSRFP,Male,0,No,No,4,Yes,Yes,DSL,No,No,No,No,Yes,No,Month-to-month,No,Electronic check,46.88,182.72,No
7114-TNGKW,Female,0,No,No,55,Yes,No,DSL,No,Yes,Yes,No,No,No,Two year,No,Mailed check,55.36,2813.56,No
7115-GZUXN,Female,0,Yes,Yes,30,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Credit card (automatic),18.57,522.21,No
7116-FANKC,Male,0,No,Yes,30,Yes,No,Fiber optic,Yes,No,No,No,Yes,No,Two year,Yes,Electronic check,77.18,2274.5,No
7117-YZJSK,Male,0,Yes,No,64,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Electronic check,18.25,1134.63,No
7118-MQKVE,Male,0,No,Yes,26,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Electronic check,20.31,527.38,No
7119-SZWBC,Male,1,No,No,53,Yes,No,Fiber optic,Yes,Yes,Yes,No,No,No,One year,Yes,Mailed check,82.78,4325.37,No
7120-RYWXD,Male,1,No,No,52,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Electronic check,18.71,953.6,No
7121-NTQUF,Female,0,Yes,Yes,29,Yes,Yes,Fiber optic,Yes,Yes,No,No,Yes,No,Two year,Yes,Bank transfer (automatic),84.3,2448.43,No
7122-LLYEH,Female,0,Yes,No,37,Yes,Yes,DSL,Yes,No,No,No,Yes,Yes,Month-to-month,No,Credit card (automatic),61.49,2108.65,No
7123-ADZWG,Male,0,No,Yes,72,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,No,Electronic check,56.81,4045.69,No
7124-SQXHZ,Female,1,Yes,No,20,Yes,Yes,DSL,Yes,Yes,No,No,Yes,No,Month-to-month,No,Bank transfer (automatic),60.71,1203.95,No
7125-JTKSJ,Female,0,Yes,No,61,No,No phone service,DSL,Yes,No,Yes,No,Yes,No,One year,No,Electronic check,57.54,3325.73,No
7126-WRGZT,Male,0,Yes,No,68,Yes,Yes,Fiber optic,No,Yes,No,No,No,Yes,Month-to-month,No,Electronic check,76.73,4902.31,No
7127-LDZUS,Female,0,Yes,No,19,No,No phone service,Fiber optic,No,Yes,Yes,No,Yes,No,Month-to-month,No,Bank transfer (automatic),79.32,1418.74,Yes
7128-VQFFB,Female,0,Yes,No,10,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Mailed check,18.25,171.52,No
7129-BTWDP,Female,0,No,No,67,Yes,Yes,Fiber optic,No,Yes,Yes,No,No,No,Month-to-month,No,Credit card (automatic),79.08,5152.01,No
7130-CKUVY,Female,0,Yes,No,9,Yes,Yes,DSL,No,No,Yes,No,No,No,Month-to-month,Yes,Mailed check,46.35,389.44,Yes
7131-HYUJH,Female,0,Yes,No,44,Yes,No,Fiber optic,No,No,No,Yes,No,Yes,Month-to-month,Yes,Electronic check,79.38,3443.57,No
7132-CYEGK,Male,0,Yes,Yes,56,Yes,Yes,Fiber optic,No,Yes,No,Yes,Yes,No,Month-to-month,Yes,Mailed check,79.64,4149.92,Yes
7133-DVSDD,Female,1,Yes,No,52,Yes,No,DSL,Yes,No,Yes,No,Yes,No,Month-to-month,No,Electronic check,51.86,2488.23,No
7134-AYLRD,Female,0,Yes,No,33,Yes,No,DSL,No,Yes,Yes,No,Yes,No,One year,No,Credit card (automatic),61.78,1940.74,No
7135-FNFQY,Male,0,No,No,65,No,No phone service,DSL,No,No,No,No,Yes,Yes,Month-to-month,No,Bank transfer (automatic),52.1,3180.55,No
7136-HQXXA,Female,0,Yes,No,21,Yes,Yes,DSL,No,Yes,Yes,No,Yes,No,One year,No,Mailed check,55.15,1180.0,No
7137-CZYHN,Male,0,Yes,No,2,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,18.25,34.69,Yes
7138-PSUJX,Male,0,Yes,Yes,58,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,No,Month-to-month,Yes,Bank transfer (automatic),80.69,4409.54,No
7139-BAYEZ,Male,1,Yes,No,53,Yes,No,Fiber optic,No,No,Yes,No,Yes,No,Month-to-month,No,Bank transfer (automatic),84.5,4208.34,No
7140-GJAYJ,Female,0,Yes,No,70,No,No phone service,DSL,No,No,No,No,No,Yes,Month-to-month,No,Electronic check,47.89,3209.44,No
7141-EMDUB,Male,1,Yes,No,49,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Electronic check,19.14,919.65,No
7142-MKABN,Male,0,No,No,8,Yes,No,Fiber optic,No,No,Yes,Yes,Yes,No,Two year,Yes,Electronic check,87.79,701.05,Yes
7143-KDDVT,Male,0,Yes,No,7,Yes,Yes,DSL,No,No,Yes,No,Yes,No,Month-to-month,Yes,Mailed check,46.76,313.07,Yes
7144-QWBSU,Female,0,No,No,11,Yes,No,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,No,Mailed check,91.37,987.58,No
7145-ENUQF,Female,0,No,No,62,Yes,No,Fiber optic,Yes,No,Yes,No,Yes,Yes,One year,Yes,Electronic check,97.78,5869.61,No
7146-RUAPN,Male,1,Yes,No,63,Yes,Yes,Fiber optic,Yes,No,Yes,No,No,No,Month-to-month,Yes,Mailed check,68.17,4314.68,No
7147-JBCUV,Male,0,Yes,No,64,Yes,No,DSL,No,No,No,No,No,Yes,Month-to-month,Yes,Mailed check,49.11,3070.45,Yes
7148-LYYRP,Female,0,No,No,4,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,67.12,254.78,No
7149-ZQYZE,Male,0,No,No,59,Yes,No,DSL,No,No,No,No,No,Yes,Month-to-month,Yes,Credit card (automatic),48.44,2830.41,No
7150-RTVJW,Female,0,Yes,Yes,1,Yes,No,DSL,No,Yes,No,No,No,Yes,Month-to-month,No,Bank transfer (automatic),52.33,52.43,Yes
7151-MKDHZ,Male,0,Yes,No,50,No,No phone service,DSL,Yes,No,No,No,No,No,One year,No,Mailed check,52.75,2549.26,No
7152-TTXGR,Female,0,Yes,Yes,11,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.55,219.08,No
7153-JWLWT,Male,0,Yes,Yes,57,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,No,Month-to-month,Yes,Electronic check,59.36,3410.28,No
7154-UYUHN,Male,0,No,No,19,Yes,Yes,Fiber optic,No,No,Yes,Yes,No,Yes,One year,Yes,Credit card (automatic),82.81,1478.51,No
7155-MBQLK,Female,0,Yes,Yes,10,Yes,Yes,DSL,No,No,Yes,Yes,No,No,One year,Yes,Bank transfer (automatic),56.06,559.86,No
7156-CNPCR,Male,0,Yes,Yes,0,No,No phone service,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Mailed check,18.25, ,Yes
7157-EDKKX,Female,0,No,Yes,11,Yes,No,Fiber optic,Yes,No,Yes,No,Yes,No,One year,Yes,Credit card (automatic),76.92,823.68,No
7158-BJTFZ,Female,0,Yes,No,72,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,No,Credit card (automatic),27.97,1956.23,No
7159-TYYCM,Female,0,Yes,No,9,Yes,Yes,DSL,Yes,No,No,No,Yes,Yes,Two year,No,Credit card (automatic),55.93,506.06,No
7160-TALSV,Female,1,No,Yes,1,Yes,Yes,DSL,Yes,No,Yes,No,No,No,Two year,Yes,Electronic check,53.45,52.44,Yes
7161-KWSBG,Female,0,Yes,No,29,Yes,No,Fiber optic,No,Yes,No,No,No,Yes,Month-to-month,Yes,Electronic check,77.41,2186.01,Yes
7162-YACJZ,Male,0,Yes,No,46,Yes,Yes,Fiber optic,No,Yes,No,Yes,No,No,Month-to-month,Yes,Bank transfer (automatic),81.19,3454.21,No
7163-SDKYF,Male,0,Yes,Yes,45,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Mailed check,27.09,1176.18,No
7164-PMUUV,Male,0,No,Yes,54,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Bank transfer (automatic),22.32,1218.3,No
7165-CLSLS,Female,0,Yes,No,27,Yes,Yes,Fiber optic,No,No,Yes,No,No,Yes,Month-to-month,Yes,Bank transfer (automatic),72.93,1951.73,Yes
7166-RJCZV,Male,0,Yes,No,40,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Electronic check,18.25,675.32,No
7167-XHHHA,Male,0,Yes,No,28,Yes,No,Fiber optic,Yes,No,No,No,Yes,No,Month-to-month,Yes,Electronic check,77.4,2185.4,Yes
7168-EHZEN,Male,0,Yes,No,1,Yes,No,DSL,No,No,No,No,Yes,No,Month-to-month,No,Mailed check,48.43,48.56,No
7169-PJYCW,Male,1,No,Yes,48,Yes,Yes,Fiber optic,Yes,No,No,Yes,No,Yes,Two year,Yes,Electronic check,76.61,3584.13,No
7170-THULZ,Male,0,Yes,No,42,Yes,No,DSL,No,No,No,Yes,No,Yes,Month-to-month,Yes,Bank transfer (automatic),57.28,2218.69,No
7171-AAPYB,Male,0,No,No,41,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Bank transfer (automatic),18.25,730.47,No
7172-UYMWU,Female,0,No,Yes,15,Yes,No,DSL,Yes,Yes,Yes,No,No,No,Two year,Yes,Mailed check,64.47,894.97,No
7173-FWSAP,Female,0,No,No,4,Yes,No,DSL,No,No,No,No,No,No,Month-to-month,Yes,Bank transfer (automatic),52.28,199.09,Yes
7174-RVUMH,Female,0,Yes,No,17,Yes,No,Fiber optic,No,No,Yes,Yes,No,Yes,Month-to-month,No,Mailed check,82.51,1378.67,Yes
7175-ECRZB,Male,0,No,No,33,Yes,No,DSL,Yes,No,No,Yes,Yes,No,Month-to-month,No,Bank transfer (automatic),60.26,2011.03,No
7176-SQEFG,Female,1,No,No,53,Yes,Yes,DSL,Yes,No,No,No,No,No,Two year,No,Electronic check,44.57,2257.75,No
7177-SHVJC,Male,0,Yes,No,11,Yes,No,Fiber optic,No,No,No,No,No,Yes,Month-to-month,Yes,Bank transfer (automatic),76.99,851.97,No
7178-CKKTM,Female,0,No,Yes,52,Yes,Yes,DSL,No,Yes,Yes,Yes,Yes,No,Month-to-month,Yes,Bank transfer (automatic),61.01,3210.07,No
7179-DFPJS,Female,0,No,Yes,25,Yes,Yes,Fiber optic,No,No,No,No,No,Yes,Month-to-month,Yes,Electronic check,70.15,1617.14,No
7180-YJPVF,Female,0,Yes,No,33,Yes,Yes,DSL,Yes,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Mailed check,66.34,2066.5,No
7181-NHRTD,Male,1,No,Yes,57,Yes,Yes,DSL,No,No,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.64,3199.38,No
7182-EAABS,Male,0,Yes,Yes,23,Yes,Yes,DSL,Yes,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,66.2,1516.03,No
7183-XKXTV,Female,0,No,Yes,10,Yes,No,Fiber optic,Yes,No,No,Yes,No,No,Two year,Yes,Bank transfer (automatic),82.8,794.25,No
7184-TENNR,Male,0,No,Yes,9,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Credit card (automatic),18.25,161.9,No
7185-VCSVC,Male,1,Yes,No,23,Yes,Yes,DSL,No,Yes,Yes,No,No,No,Month-to-month,Yes,Credit card (automatic),60.94,1375.29,No
7186-KXADV,Male,0,No,No,2,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,18.25,36.28,No
7187-ESXMM,Male,1,Yes,No,19,Yes,No,Fiber optic,No,No,Yes,Yes,No,No,Month-to-month,Yes,Bank transfer (automatic),85.46,1610.53,No
7188-LJKPX,Male,0,No,Yes,0,Yes,Yes,Fiber optic,No,No,No,No,No,Yes,Month-to-month,Yes,Credit card (automatic),70.43,0.0,Yes
7189-ZWCHU,Male,1,No,No,60,Yes,Yes,Fiber optic,No,Yes,Yes,No,No,No,Two year,Yes,Mailed check,79.0,4435.81,No
7190-PJHCH,Female,0,Yes,No,37,Yes,Yes,Fiber optic,No,Yes,No,No,No,No,Month-to-month,No,Electronic check,72.54,2504.89,No
7191-LKPGE,Female,0,Yes,No,27,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Electronic check,18.63,509.67,Yes
7192-XKGRP,Female,0,Yes,No,35,Yes,Yes,Fiber optic,No,No,No,Yes,No,Yes,Two year,No,Credit card (automatic),82.21,2910.02,No
7193-PFTEW,Male,0,Yes,No,18,Yes,Yes,Fiber optic,No,No,Yes,No,No,No,Month-to-month,Yes,Mailed check,80.5,1448.63,Yes
7194-LKKLR,Female,0,Yes,No,67,Yes,No,DSL,Yes,Yes,No,No,Yes,No,Month-to-month,Yes,Electronic check,63.26,3948.82,No
7195-MDPGC,Male,0,Yes,Yes,27,No,No phone service,DSL,Yes,No,No,No,No,Yes,Month-to-month,No,Mailed check,52.38,1382.34,No
7196-WHQEJ,Male,0,Yes,Yes,25,Yes,Yes,Fiber optic,No,No,No,Yes,No,No,Month-to-month,Yes,Bank transfer (automatic),75.24,1765.74,Yes
7197-KDZHP,Male,0,Yes,No,25,Yes,Yes,DSL,Yes,Yes,No,No,Yes,Yes,Month-to-month,Yes,Mailed check,59.87,1432.15,Yes
7198-PLQPB,Male,0,No,Yes,0,Yes,No,DSL,No,No,No,No,No,Yes,One year,Yes,Credit card (automatic),45.53,0.0,No
7199-SCLJL,Female,0,No,No,53,Yes,Yes,DSL,No,Yes,Yes,No,Yes,No,Month-to-month,Yes,Bank transfer (automatic),53.91,2865.5,No
7200-BSLBA,Male,0,No,No,13,Yes,No,DSL,Yes,No,No,No,Yes,No,One year,Yes,Credit card (automatic),52.25,634.72,No
7201-NRPTU,Female,1,Yes,No,11,Yes,Yes,Fiber optic,Yes,No,Yes,No,Yes,Yes,Month-to-month,No,Credit card (automatic),89.0,963.94,Yes
7202-WZXVL,Male,0,No,No,16,Yes,Yes,Fiber optic,No,No,No,Yes,No,Yes,Month-to-month,Yes,Electronic check,71.35,1162.59,Yes
7203-NQESQ,Female,0,No,No,13,Yes,No,Fiber optic,No,No,Yes,No,No,Yes,Month-to-month,Yes,Bank transfer (automatic),78.81,1012.96,Yes
7204-NWSYZ,Male,1,Yes,Yes,16,Yes,Yes,DSL,No,Yes,Yes,No,Yes,No,One year,Yes,Electronic check,64.56,958.51,No
7205-VHEHT,Male,1,Yes,Yes,5,Yes,Yes,Fiber optic,No,Yes,No,No,No,No,Month-to-month,No,Credit card (automatic),78.99,399.21,Yes
7206-WXSSY,Female,0,No,No,67,No,No phone service,Fiber optic,Yes,Yes,No,Yes,Yes,No,Month-to-month,Yes,Electronic check,85.66,5649.16,No
7207-SDNJR,Female,0,Yes,No,34,Yes,No,Fiber optic,No,No,No,Yes,No,Yes,Two year,Yes,Bank transfer (automatic),73.41,2410.16,No
7208-RHHCH,Male,0,No,Yes,37,Yes,No,Fiber optic,Yes,No,No,No,No,No,One year,No,Credit card (automatic),76.34,2784.95,No
7209-VRYYW,Female,1,No,No,9,No,No phone service,Fiber optic,No,Yes,Yes,Yes,No,No,Month-to-month,No,Electronic check,74.23, ,Yes
7210-EJBCF,Male,0,Yes,No,44,Yes,No,Fiber optic,No,Yes,Yes,No,No,No,Two year,No,Credit card (automatic),83.44,3552.82,No
7211-GAWNR,Female,0,Yes,No,10,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Electronic check,19.15,186.35,Yes
7212-RZDJH,Female,0,No,No,65,Yes,No,Fiber optic,No,No,No,Yes,No,Yes,Month-to-month,Yes,Mailed check,86.33,5416.42,No
7213-PNUPF,Male,1,No,No,21,Yes,Yes,Fiber optic,Yes,No,No,No,Yes,No,Month-to-month,Yes,Mailed check,72.09,1403.77,Yes
7214-PZPQE,Female,0,Yes,Yes,14,Yes,No,Fiber optic,Yes,No,No,No,Yes,No,Month-to-month,No,Mailed check,82.67,1155.88,Yes
7215-AFALX,Male,0,No,No,47,Yes,No,DSL,No,No,No,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),53.6,2349.51,No
7216-LRWUR,Male,0,No,No,3,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Credit card (automatic),18.68,53.77,No
7217-UGLKV,Male,0,No,No,41,Yes,No,DSL,Yes,No,No,Yes,No,Yes,Month-to-month,Yes,Electronic check,53.02,2084.0,No
7218-KYEET,Male,0,No,No,3,No,No phone service,DSL,No,No,No,Yes,No,Yes,Two year,No,Bank transfer (automatic),56.38,160.35,No
7219-USHAH,Female,1,Yes,Yes,35,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Credit card (automatic),22.7,762.92,No
7220-MKYWU,Male,0,Yes,Yes,6,Yes,Yes,Fiber optic,Yes,No,Yes,No,Yes,No,One year,Yes,Bank transfer (automatic),90.64,533.94,Yes
7221-ADLQS,Male,0,No,Yes,41,Yes,No,DSL,Yes,Yes,No,No,No,No,One year,No,Electronic check,58.03,2262.98,No
7222-HCHVZ,Female,0,Yes,No,1,No,No phone service,DSL,No,No,No,No,No,No,One year,No,Credit card (automatic),42.62,39.95,Yes
7223-YQCZD,Male,0,No,No,6,No,No phone service,DSL,No,No,Yes,Yes,Yes,No,Two year,Yes,Electronic check,54.48,329.15,No
7224-GZNEN,Female,0,No,Yes,36,Yes,No,Fiber optic,No,No,No,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),73.53,2692.3,No
7225-BHYHX,Male,0,Yes,No,40,Yes,No,DSL,No,No,Yes,Yes,Yes,No,Month-to-month,No,Bank transfer (automatic),64.39,2487.66,No
7226-PRSPA,Female,0,No,No,66,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Bank transfer (automatic),18.25,1147.53,No
7227-YMRPR,Female,1,Yes,No,37,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Electronic check,25.92,886.87,No
7228-JWFRP,Female,0,No,No,17,Yes,No,Fiber optic,No,No,No,No,Yes,No,Month-to-month,Yes,Electronic check,76.6,1202.82,No
7229-NAKBN,Male,0,Yes,Yes,15,Yes,No,Fiber optic,No,No,Yes,No,No,No,One year,No,Bank transfer (automatic),75.82,1091.0,Yes
7230-SNUJH,Male,1,No,Yes,11,Yes,No,DSL,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,57.27,638.17,Yes
7231-LFVVK,Female,0,No,No,69,Yes,Yes,DSL,No,Yes,Yes,No,No,Yes,Two year,No,Mailed check,61.14,4189.43,No
7232-VADGK,Male,0,Yes,Yes,9,Yes,Yes,Fiber optic,No,No,No,No,Yes,No,Month-to-month,Yes,Bank transfer (automatic),74.44,639.76,Yes
7233-FNVLL,Female,0,No,No,43,Yes,No,DSL,No,No,Yes,No,No,No,Month-to-month,Yes,Bank transfer (automatic),48.02,2099.56,Yes
7234-BGVVC,Male,0,No,Yes,68,Yes,Yes,DSL,No,Yes,No,No,No,Yes,Month-to-month,No,Electronic check,56.78,3717.7,No
7235-AVEPV,Female,0,No,Yes,4,Yes,No,DSL,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,57.92,219.56,Yes
7236-UPKUS,Female,0,No,Yes,2,Yes,No,DSL,No,No,Yes,No,Yes,No,Month-to-month,Yes,Credit card (automatic),50.81,97.82,Yes
7237-WNYUB,Male,0,No,No,10,Yes,Yes,Fiber optic,No,No,No,Yes,No,Yes,Month-to-month,Yes,Mailed check,81.4,766.1,Yes
7238-HVCTJ,Male,0,Yes,Yes,5,Yes,Yes,Fiber optic,No,No,No,Yes,No,Yes,Month-to-month,No,Electronic check,78.74,397.07,Yes
7239-ENYXK,Female,0,No,Yes,34,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Electronic check,21.96,713.88,No
7240-AXXQR,Male,1,Yes,No,4,No,No phone service,Fiber optic,No,No,No,No,Yes,No,Month-to-month,Yes,Mailed check,72.68,286.59,Yes
7241-NADDX,Female,0,No,No,7,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Credit card (automatic),18.25,122.5,No
7242-BPJAY,Male,0,No,No,38,No,No phone service,Fiber optic,No,No,No,Yes,No,No,Month-to-month,Yes,Electronic check,81.44,3126.2,No
7243-AWUVD,Female,0,Yes,Yes,2,No,No phone service,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Electronic check,26.19,51.55,No
7244-XQKSJ,Female,0,Yes,No,2,Yes,Yes,Fiber optic,No,No,Yes,No,No,No,Month-to-month,Yes,Mailed check,78.04,151.21,Yes
7245-GGPQF,Male,0,No,Yes,59,Yes,No,DSL,No,No,No,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,56.95,3135.65,No
7246-FYNWH,Female,1,Yes,No,3,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Electronic check,18.8,56.97,Yes
7247-GGNUN,Male,0,No,Yes,55,Yes,Yes,DSL,No,No,Yes,No,No,No,One year,No,Credit card (automatic),55.03,2950.02,No
7248-HZPKA,Female,0,Yes,No,10,Yes,No,DSL,No,No,No,No,Yes,No,Two year,No,Bank transfer (automatic),48.08,456.04,No
7249-PQYFD,Male,0,No,No,1,Yes,No,DSL,No,No,No,Yes,Yes,No,Two year,Yes,Electronic check,51.47,51.87,No
7250-ZPACM,Male,0,Yes,Yes,68,No,No phone service,Fiber optic,Yes,No,No,No,Yes,Yes,Month-to-month,Yes,Mailed check,90.99,5726.02,No
7251-QFJTZ,Male,1,Yes,Yes,18,Yes,No,Fiber optic,No,Yes,Yes,Yes,No,No,Month-to-month,Yes,Electronic check,83.31,1426.71,No
7252-MKJDB,Male,0,No,Yes,8,Yes,Yes,Fiber optic,Yes,No,No,No,No,No,Month-to-month,No,Electronic check,81.57,635.39,Yes
7253-WAVFS,Male,0,No,No,4,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Bank transfer (automatic),18.25,69.64,No
7254-YTTGQ,Female,0,Yes,Yes,65,Yes,Yes,DSL,No,No,Yes,No,No,No,One year,Yes,Mailed check,51.57,3243.69,No
7255-RLAXE,Male,0,Yes,No,1,Yes,Yes,Fiber optic,No,No,No,Yes,Yes,Yes,Two year,Yes,Bank transfer (automatic),91.78,86.86,Yes
7256-ZYATR,Female,1,No,No,32,Yes,No,Fiber optic,Yes,No,Yes,No,Yes,No,Two year,Yes,Credit card (automatic),83.62,2496.58,No
7257-FVCFY,Male,0,No,No,40,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Electronic check,18.94,699.78,No
7258-TCDGB,Male,0,Yes,No,41,Yes,No,DSL,No,Yes,No,No,No,No,One year,Yes,Credit card (automatic),44.83,1719.96,No
7259-EPAGH,Female,0,No,No,67,Yes,Yes,DSL,No,No,No,No,No,No,Two year,No,Mailed check,49.33,3355.66,No
7260-ZZWJN,Female,0,No,Yes,49,Yes,Yes,Fiber optic,Yes,No,No,No,Yes,No,Month-to-month,No,Electronic check,75.92,3636.53,No
7261-NWTXP,Male,0,No,No,3,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Two year,No,Credit card (automatic),87.62,254.68,No
7262-YTTRJ,Male,1,No,No,26,No,No phone service,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Electronic check,18.25,454.78,No
7263-QJRGV,Female,0,No,No,20,Yes,No,Fiber optic,No,No,Yes,No,No,No,Two year,Yes,Mailed check,80.73,1599.81,Yes
7264-PTFVV,Female,0,Yes,No,3,Yes,No,Fiber optic,No,No,No,No,No,No,One year,No,Bank transfer (automatic),74.62,209.11,Yes
7265-DJSWT,Female,0,No,No,9,Yes,No,Fiber optic,Yes,No,Yes,No,Yes,No,Month-to-month,No,Electronic check,83.37,711.02,Yes
7266-MHXWX,Male,0,No,Yes,28,Yes,Yes,DSL,Yes,Yes,Yes,Yes,No,Yes,One year,No,Credit card (automatic),62.88,1735.27,No
7267-HLFYW,Male,1,No,No,7,Yes,No,DSL,No,No,No,No,No,Yes,Month-to-month,No,Electronic check,55.64,376.98,Yes
7268-EFPKM,Female,0,Yes,No,63,Yes,Yes,DSL,No,No,Yes,Yes,No,No,One year,Yes,Credit card (automatic),52.54,3300.12,No
7269-JSXYL,Female,0,Yes,No,27,No,No phone service,DSL,No,No,No,No,No,No,Two year,Yes,Electronic check,40.58,1106.3,No
7270-VSEVJ,Male,0,No,No,22,Yes,No,Fiber optic,No,No,Yes,Yes,No,Yes,One year,Yes,Electronic check,76.45,1685.83,No
7271-RTYYV,Male,0,Yes,No,2,Yes,Yes,Fiber optic,No,Yes,No,Yes,No,No,Two year,Yes,Mailed check,76.42,143.43,No
7272-NKLYE,Male,0,No,No,24,Yes,Yes,Fiber optic,Yes,No,No,No,No,Yes,Month-to-month,Yes,Credit card (automatic),73.18,1631.88,Yes
7273-NSARG,Female,0,Yes,No,3,Yes,Yes,Fiber optic,Yes,Yes,No,Yes,No,No,Month-to-month,Yes,Electronic check,81.01,227.71,Yes
7274-GQADL,Male,0,No,Yes,29,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Credit card (automatic),86.48,2503.79,Yes
7275-QCJHN,Male,0,No,No,11,Yes,No,Fiber optic,Yes,Yes,No,No,No,Yes,Month-to-month,No,Electronic check,82.0,912.27,No
7276-HANWQ,Male,0,Yes,Yes,7,Yes,No,DSL,No,No,No,Yes,No,Yes,Month-to-month,No,Mailed check,55.42,392.81,No
7277-JMDFL,Female,0,Yes,Yes,6,Yes,No,Fiber optic,No,Yes,Yes,Yes,Yes,No,Month-to-month,Yes,Mailed check,85.11,485.04,Yes
7278-QMZRA,Male,0,No,No,8,Yes,Yes,Fiber optic,Yes,No,Yes,No,Yes,No,Month-to-month,Yes,Electronic check,83.76,672.78,Yes
7279-NPBKW,Male,0,No,No,2,Yes,No,Fiber optic,Yes,No,No,Yes,No,No,Month-to-month,Yes,Credit card (automatic),86.68,168.79,Yes
7280-JLFBH,Male,0,Yes,No,3,Yes,Yes,DSL,Yes,No,Yes,Yes,No,No,Two year,No,Mailed check,57.11,171.72,No
7281-UZRVR,Male,0,No,No,33,Yes,No,DSL,No,Yes,No,Yes,No,No,Month-to-month,No,Bank transfer (automatic),55.54,1786.7,No
7282-RLBKW,Female,0,No,No,25,Yes,Yes,Fiber optic,No,Yes,No,No,No,Yes,Two year,No,Bank transfer (automatic),72.74,1699.55,No
7283-NUBEH,Male,0,Yes,No,10,Yes,Yes,Fiber optic,No,No,Yes,No,No,No,Month-to-month,Yes,Mailed check,80.58,747.72,Yes
7284-JZMRR,Male,0,Yes,No,27,Yes,No,DSL,No,No,No,Yes,No,Yes,Month-to-month,Yes,Bank transfer (automatic),52.09,1428.54,No
7285-LZFAT,Female,0,Yes,No,8,No,No phone service,Fiber optic,No,No,No,No,Yes,No,Month-to-month,No,Credit card (automatic),78.7,606.04,Yes
7286-RGWDG,Male,0,Yes,No,38,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Bank transfer (automatic),19.51,708.06,No
7287-DETSR,Female,0,Yes,No,39,Yes,No,DSL,No,No,No,Yes,No,No,Month-to-month,Yes,Mailed check,51.95,1936.15,No
7288-WBQCY,Female,0,Yes,No,63,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Bank transfer (automatic),22.68,1368.68,No
7289-RUPKS,Male,0,No,No,59,Yes,No,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Bank transfer (automatic),47.69,2854.51,No
7290-WEWTC,Female,0,Yes,No,25,Yes,No,Fiber optic,No,No,No,No,No,Yes,Month-to-month,Yes,Mailed check,78.87,1826.37,Yes
7291-YEQDE,Male,0,No,No,11,Yes,No,Fiber optic,No,No,Yes,No,No,Yes,One year,No,Bank transfer (automatic),75.96,778.66,No
7292-RSSJV,Female,0,Yes,No,35,Yes,No,DSL,No,No,No,No,Yes,No,Month-to-month,No,Mailed check,52.61,1839.44,No
7293-BWHQX,Male,0,Yes,No,65,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Bank transfer (automatic),18.44,1168.56,No
7294-GKVWF,Female,0,No,No,49,Yes,No,Fiber optic,No,No,No,No,Yes,No,Month-to-month,Yes,Credit card (automatic),75.9,3469.38,No
7295-NHNKA,Female,0,No,No,52,No,No phone service,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Credit card (automatic),18.25,876.35,No
7296-YRMCB,Female,0,Yes,Yes,34,Yes,No,DSL,Yes,No,No,No,Yes,No,One year,Yes,Credit card (automatic),54.2,1736.7,No
7297-MRGTK,Male,0,Yes,Yes,6,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Bank transfer (automatic),18.25,103.6,No
7298-ANEAD,Male,0,Yes,No,72,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Bank transfer (automatic),20.99,1392.53,No
7299-GNZFB,Male,0,No,No,57,Yes,No,Fiber optic,Yes,No,No,Yes,Yes,No,Month-to-month,No,Electronic check,83.68,4491.25,No
7300-EHEHZ,Male,0,Yes,Yes,67,Yes,Yes,DSL,No,No,No,No,Yes,Yes,Month-to-month,Yes,Mailed check,51.84,3451.86,No
7301-KRSDF,Female,1,Yes,No,18,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,No,Electronic check,69.78,1263.0,Yes
7302-KMARH,Male,1,Yes,No,55,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,No,Two year,No,Electronic check,79.54,4417.68,No
7303-HQYWZ,Female,0,No,Yes,54,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Credit card (automatic),18.25,945.09,No
7304-KCFJH,Female,0,No,No,2,Yes,Yes,Fiber optic,No,Yes,No,No,No,No,Month-to-month,No,Credit card (automatic),73.98,140.85,Yes
7305-EDERM,Male,0,No,No,9,Yes,No,DSL,No,No,No,Yes,No,No,Two year,No,Mailed check,45.48,417.5,No
7306-APWLH,Male,0,No,No,63,Yes,No,DSL,Yes,Yes,Yes,Yes,No,No,Month-to-month,No,Electronic check,72.38,4545.96,No
7307-GKGYZ,Female,0,Yes,No,68,Yes,No,Fiber optic,No,No,No,No,No,No,One year,Yes,Bank transfer (automatic),74.24,4915.73,No
7308-JYWBM,Male,0,Yes,Yes,5,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,21.09,99.7,No
7309-RLLMS,Male,0,Yes,No,2,Yes,No,DSL,Yes,No,Yes,No,No,Yes,Month-to-month,Yes,Credit card (automatic),63.19,120.33,Yes
7310-HQFXC,Female,0,No,No,40,Yes,No,Fiber optic,No,Yes,Yes,No,No,No,One year,Yes,Credit card (automatic),71.08,2762.27,No
7311-BAJQZ,Female,1,No,No,48,No,No phone service,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Mailed check,18.25,853.41,No
7312-LEJDM,Male,0,No,No,3,Yes,No,DSL,No,Yes,No,Yes,Yes,No,Month-to-month,No,Electronic check,56.9,157.28,Yes
7313-TAUFS,Male,0,No,Yes,1,Yes,Yes,DSL,No,No,Yes,No,Yes,No,Month-to-month,Yes,Electronic check,48.49,49.26,Yes
7314-VUSSE,Male,0,No,Yes,2,No,No phone service,Fiber optic,No,Yes,Yes,No,No,No,Month-to-month,Yes,Electronic check,78.05,152.6,Yes
7315-DKQQU,Female,0,No,No,27,No,No phone service,Fiber optic,Yes,Yes,No,No,Yes,No,One year,No,Mailed check,84.32,2228.64,No
7316-HWLFT,Female,0,Yes,No,70,Yes,No,DSL,Yes,Yes,No,Yes,No,No,Month-to-month,Yes,Electronic check,62.18,4096.44,No
7317-DKUUG,Female,0,Yes,Yes,38,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Electronic check,18.25,691.07,No
7318-DACVU,Female,0,Yes,No,31,Yes,Yes,Fiber optic,No,Yes,No,No,No,No,Two year,Yes,Electronic check,76.1,2344.3,No
7319-NPHEA,Female,0,No,No,11,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Electronic check,20.06,205.11,No
7320-SACVP,Male,1,Yes,No,37,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Bank transfer (automatic),21.39,775.21,No
7321-MFDMJ,Male,0,No,No,60,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,No,Electronic check,18.25,1079.47,No
7322-NDFCL,Female,0,No,Yes,0,Yes,No,Fiber optic,No,Yes,No,No,Yes,No,One year,Yes,Credit card (automatic),84.53,0.0,Yes
7323-GFPMF,Female,0,No,No,31,Yes,Yes,DSL,No,Yes,Yes,No,Yes,No,Month-to-month,Yes,Electronic check,58.09,1693.84,No
7324-SJJMD,Female,0,Yes,No,66,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,90.6,6026.02,No
7325-LPDPA,Male,0,Yes,No,29,Yes,No,DSL,No,Yes,No,No,No,No,Month-to-month,No,Electronic check,46.34,1298.37,No
7326-QAEJQ,Male,1,Yes,No,18,Yes,Yes,Fiber optic,No,No,No,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),81.97,1371.66,No
7327-TMDKU,Male,1,No,No,20,Yes,No,DSL,Yes,No,Yes,No,No,Yes,Month-to-month,No,Electronic check,58.26,1084.29,No
7328-CTKAS,Male,0,No,Yes,41,Yes,No,Fiber optic,No,No,No,Yes,Yes,No,Two year,Yes,Mailed check,77.69,2993.47,No
7329-AJEQB,Female,0,No,Yes,4,Yes,No,DSL,Yes,Yes,No,No,Yes,No,Month-to-month,No,Electronic check,61.03,231.49,Yes
7330-CNZAM,Female,0,Yes,No,45,Yes,Yes,DSL,No,No,Yes,Yes,No,No,Two year,Yes,Credit card (automatic),52.8,2303.87,No
7331-VGMWT,Male,0,Yes,No,59,Yes,No,DSL,No,Yes,Yes,No,No,No,Month-to-month,Yes,Electronic check,43.35,2544.54,No
7332-HUSNL,Male,0,Yes,No,38,Yes,No,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Mailed check,71.25,2541.46,Yes
7333-YWZSJ,Female,0,No,No,18,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Credit card (automatic),25.09,460.32,No
7334-CDYJC,Male,0,No,No,55,Yes,Yes,DSL,No,No,No,No,Yes,Yes,Two year,Yes,Electronic check,58.92,3033.69,No
7335-FANQK,Female,0,Yes,No,65,No,No phone service,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Electronic check,27.55,1754.82,No
7336-TMZUB,Female,0,No,Yes,59,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Credit card (automatic),19.93,1191.03,No
7337-GUNUJ,Male,0,Yes,No,6,No,No phone service,Fiber optic,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,69.33,417.33,No
7338-AUURK,Male,0,No,No,6,Yes,No,Fiber optic,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,72.82,430.27,Yes
7339-ATRSE,Female,0,Yes,No,63,No,No phone service,Fiber optic,Yes,Yes,No,No,Yes,Yes,Month-to-month,Yes,Mailed check,88.05,5500.27,No
7340-TTWPT,Female,1,Yes,Yes,18,Yes,Yes,DSL,No,Yes,Yes,No,No,Yes,Two year,Yes,Electronic check,54.47,915.18,No
7341-KSSUW,Male,0,No,No,9,Yes,Yes,Fiber optic,No,No,No,No,No,Yes,Month-to-month,Yes,Credit card (automatic),81.09,672.13,Yes
7342-AUVPN,Female,0,Yes,No,29,Yes,Yes,DSL,Yes,No,No,No,No,No,One year,Yes,Bank transfer (automatic),58.46,1633.32,No
7343-TLKVU,Female,1,Yes,No,40,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Bank transfer (automatic),21.91,814.13,No
7344-EVHDX,Female,0,Yes,No,70,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Electronic check,19.96,1335.41,No
7345-FEGMD,Male,0,Yes,No,30,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Bank transfer (automatic),18.25,515.95,No
7346-DKFEX,Female,0,No,No,1,Yes,No,Fiber optic,No,No,Yes,Yes,Yes,No,Month-to-month,No,Bank transfer (automatic),89.69,90.73,Yes
7347-QACVG,Female,0,No,No,5,Yes,Yes,Fiber optic,No,Yes,No,No,No,Yes,One year,Yes,Electronic check,79.31,399.72,No
7348-DRWKT,Male,0,No,No,20,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Credit card (automatic),18.84,377.51,Yes
7349-LCVYZ,Male,0,Yes,Yes,72,Yes,Yes,DSL,No,No,Yes,No,No,No,Two year,Yes,Bank transfer (automatic),51.42,3670.38,No
7350-CPQBY,Female,0,No,No,61,Yes,No,DSL,No,Yes,No,No,No,No,One year,Yes,Credit card (automatic),47.16,2690.94,No
7351-CGZPS,Male,0,Yes,No,39,Yes,No,Fiber optic,No,Yes,Yes,Yes,Yes,Yes,Two year,Yes,Electronic check,93.09,3357.63,No
7352-MUBSB,Male,0,Yes,Yes,1,Yes,No,DSL,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,39.23,38.66,Yes
7353-ENLDV,Male,1,Yes,No,1,Yes,Yes,DSL,Yes,No,No,Yes,Yes,No,Month-to-month,No,Bank transfer (automatic),62.85,61.86,Yes
7354-HWAFH,Female,0,No,No,6,Yes,Yes,DSL,No,No,Yes,No,No,No,Month-to-month,No,Credit card (automatic),49.08,281.8,Yes
7355-QKDPY,Female,0,No,Yes,18,Yes,No,Fiber optic,Yes,No,Yes,No,Yes,Yes,Month-to-month,Yes,Mailed check,92.64,1612.94,Yes
7356-DGMTD,Male,0,No,No,4,Yes,Yes,DSL,No,No,Yes,Yes,Yes,Yes,Month-to-month,No,Electronic check,62.2,241.07,Yes
7357-FFQBR,Female,0,No,No,3,Yes,No,DSL,No,No,Yes,Yes,No,Yes,Month-to-month,Yes,Credit card (automatic),57.96,164.14,Yes
7358-LHWVV,Male,0,No,No,41,Yes,No,DSL,No,No,No,No,No,Yes,Month-to-month,Yes,Electronic check,44.65,1741.71,No
7359-XFWEH,Female,0,No,Yes,62,Yes,No,Fiber optic,No,No,Yes,No,No,No,Month-to-month,No,Electronic check,77.54,4624.02,No
7360-PTPPK,Female,0,No,No,14,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Bank transfer (automatic),18.25,250.47,Yes
7361-QMMQQ,Female,0,Yes,No,40,Yes,No,DSL,No,Yes,No,No,Yes,No,Month-to-month,Yes,Electronic check,52.05,2114.44,No
7362-HDHKD,Male,1,Yes,No,13,Yes,No,Fiber optic,No,No,No,Yes,No,No,Month-to-month,No,Electronic check,74.5,943.17,No
7363-EYGTD,Female,0,Yes,Yes,34,Yes,No,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,No,Electronic check,88.48,2987.61,No
7364-ZFKUE,Male,0,No,No,8,Yes,No,DSL,Yes,Yes,Yes,Yes,No,Yes,Month-to-month,No,Credit card (automatic),60.68,485.78,Yes
7365-PUAAU,Female,0,No,No,52,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Electronic check,19.08,924.05,No
7366-GXYXZ,Male,0,No,Yes,27,No,No phone service,Fiber optic,No,Yes,No,Yes,Yes,No,Month-to-month,No,Bank transfer (automatic),86.68,2305.99,No
7367-VKZJQ,Male,0,Yes,Yes,31,Yes,Yes,Fiber optic,Yes,Yes,No,No,No,No,One year,Yes,Credit card (automatic),77.49,2325.83,No
7368-AZQYJ,Male,0,No,Yes,69,Yes,No,DSL,No,Yes,Yes,No,No,Yes,Month-to-month,Yes,Bank transfer (automatic),57.6,3883.95,Yes
7369-HNWPV,Female,0,Yes,No,12,Yes,No,DSL,No,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,63.65,705.63,Yes
7370-FUTXB,Male,0,Yes,Yes,11,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.5,216.78,No
7371-KCKUR,Female,1,Yes,No,52,No,No phone service,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,18.25,963.5,No
7372-UTZXQ,Male,0,Yes,No,53,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Bank transfer (automatic),19.85,968.25,No
7373-XBTNY,Male,0,No,No,15,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Credit card (automatic),23.98,338.58,Yes
7374-NETYA,Female,1,Yes,No,38,Yes,Yes,DSL,No,No,No,No,Yes,No,Month-to-month,Yes,Bank transfer (automatic),45.3,1614.71,Yes
7375-BRPPC,Male,0,No,Yes,1,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Mailed check,18.5,18.47,Yes
7376-PPWKJ,Female,1,Yes,No,1,Yes,No,Fiber optic,No,No,No,Yes,No,No,One year,No,Electronic check,68.97,67.58,No
7377-BPRGT,Female,0,Yes,No,20,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Credit card (automatic),23.77,445.8,No
7378-YEPSA,Male,1,No,No,17,Yes,Yes,DSL,No,Yes,No,No,Yes,No,Month-to-month,Yes,Electronic check,53.63,912.98,Yes
7379-EKDJD,Male,0,Yes,No,6,Yes,No,DSL,No,No,Yes,Yes,No,No,Month-to-month,No,Credit card (automatic),50.6,290.04,Yes
7380-XQHUA,Male,0,No,No,2,Yes,Yes,Fiber optic,No,No,No,No,Yes,No,One year,Yes,Electronic check,74.87,144.66,Yes
7381-QSVFQ,Female,0,No,Yes,41,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Bank transfer (automatic),18.25,717.37,No
7382-UVEZG,Female,0,No,No,37,Yes,Yes,Fiber optic,No,No,No,No,No,No,One year,Yes,Credit card (automatic),65.87,2369.29,No
7383-KEDGD,Male,0,Yes,No,70,Yes,Yes,DSL,Yes,No,No,No,Yes,No,Month-to-month,No,Mailed check,48.16,3263.37,No
7384-TQKPQ,Male,1,Yes,No,37,Yes,No,DSL,No,No,No,Yes,No,Yes,One year,Yes,Electronic check,49.73,1743.33,No
7385-EHXAY,Male,0,No,Yes,45,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Credit card (automatic),25.43,1101.39,No
7386-ZGHEP,Male,0,Yes,No,52,Yes,Yes,Fiber optic,No,No,Yes,No,No,No,One year,Yes,Mailed check,71.57,3761.96,No
7387-FUKZW,Female,0,Yes,No,20,Yes,No,Fiber optic,Yes,No,Yes,No,Yes,Yes,One year,Yes,Electronic check,89.86,1743.8,No
7388-KQABN,Female,0,Yes,No,6,Yes,No,Fiber optic,Yes,No,Yes,No,No,Yes,One year,No,Mailed check,80.15,486.5,Yes
7389-MKCWK,Male,0,No,Yes,7,Yes,No,Fiber optic,Yes,No,Yes,No,Yes,Yes,Two year,Yes,Mailed check,91.04,621.36,No
7390-MKMRU,Female,1,No,No,33,Yes,No,Fiber optic,No,Yes,No,No,No,Yes,Two year,Yes,Bank transfer (automatic),74.9,2463.99,No
7391-PDWGA,Male,0,No,No,51,Yes,No,Fiber optic,No,Yes,Yes,Yes,Yes,No,Month-to-month,No,Electronic check,88.16,4268.47,Yes
7392-AUKAV,Male,0,No,Yes,7,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,20.21,141.33,No
7393-HZXBS,Female,0,Yes,Yes,45,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Credit card (automatic),29.31,1248.3,No
7394-QEJMD,Female,0,No,No,48,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Electronic check,22.52,996.6,No
7395-TGYVR,Female,0,Yes,Yes,63,Yes,Yes,Fiber optic,No,No,No,No,No,Yes,One year,Yes,Mailed check,72.13,4253.69,No
7396-CADMU,Male,0,No,No,11,Yes,Yes,Fiber optic,No,Yes,No,Yes,Yes,Yes,Month-to-month,No,Bank transfer (automatic),90.74,920.45,Yes
7397-DSBGQ,Male,0,Yes,No,2,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Credit card (automatic),92.05,186.81,Yes
7398-VUNAH,Male,1,No,No,46,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Bank transfer (automatic),18.25,856.29,No
7399-UBSSU,Male,0,Yes,No,64,Yes,No,Fiber optic,No,Yes,No,Yes,Yes,No,One year,Yes,Credit card (automatic),80.13,5108.95,No
"""

df = pd.read_csv(io.StringIO(datos_csv))

# Revisamos que se haya cargado bien
print(f"Filas: {df.shape[0]}   Columnas: {df.shape[1]}")
df.head()


In [ ]:
# INTERPRETACIÓN
print("Interpretación:")
print(f"Cargamos {df.shape[0]} clientes con {df.shape[1]} variables cada uno.")
print("Esto es exactamente la misma estructura que Telco Customer Churn del Caso I:")
print("cada fila es un cliente, y la columna 'Churn' nos dice si canceló (Yes) o no (No).")


## Parte 1 · Reconstruir el modelo de churn

Vamos a repetir, de forma ágil, el mismo flujo del Caso I: limpiar datos, codificar variables categóricas, entrenar una regresión logística y evaluarla.


In [ ]:
# TotalCharges a veces llega como texto (por ejemplo, con un espacio en vez de un número).
# pd.to_numeric() lo convierte a número; errors='coerce' pone NaN donde no se puede convertir.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Si algún cliente quedó con NaN (por el dato sucio), lo llenamos con 0:
# tiene sentido de negocio porque esos son clientes con tenure = 0 (recién llegados).
df["TotalCharges"] = df["TotalCharges"].fillna(0)

print("Valores nulos después de la limpieza:")
print(df.isnull().sum().sum())


In [ ]:
# INTERPRETACIÓN
print("Interpretación:")
print("TotalCharges ya es una columna numérica. Este paso es el mismo error frecuente")
print("que vimos en el Caso I: si no se convierte, el modelo la ignora o el código truena.")


In [ ]:
# Variable objetivo: convertimos 'Yes'/'No' en 1/0 para que el modelo la entienda
df["Churn_bin"] = (df["Churn"] == "Yes").astype(int)

# Seleccionamos variables numéricas simples + codificamos Contract (la más importante según la clase)
# pd.get_dummies convierte una columna de texto en columnas 0/1, una por categoría
df_modelo = df[["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen", "Contract", "Churn_bin"]].copy()
df_modelo = pd.get_dummies(df_modelo, columns=["Contract"], drop_first=True)

print("Variables que va a usar el modelo:")
print(df_modelo.drop(columns=["Churn_bin"]).columns.tolist())


In [ ]:
# Dividimos en train (80%) y test (20%), igual que en el Caso I
X = df_modelo.drop(columns=["Churn_bin"])
y = df_modelo["Churn_bin"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Escalamos SOLO con la información de train, para no "hacer trampa" con el test
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"Clientes de entrenamiento: {X_train.shape[0]}   Clientes de prueba: {X_test.shape[0]}")


In [ ]:
# Entrenamos la regresión logística
modelo = LogisticRegression(max_iter=1000, random_state=42)
modelo.fit(X_train_s, y_train)

# Predecimos sobre el set de prueba
y_pred = modelo.predict(X_test_s)

print("Reporte de clasificación:")
print(classification_report(y_test, y_pred, target_names=["No cancela", "Cancela"]))


In [ ]:
# INTERPRETACIÓN
print("Interpretación:")
print("El 'recall' de la clase 'Cancela' es el número más importante para el negocio:")
print("nos dice qué porcentaje de los clientes que SÍ se iban a ir, el modelo logró detectar.")
print("Un recall bajo en esa clase significa que se nos están escapando clientes de riesgo real.")


## Parte 2 · Curva ROC, AUC y punto de corte óptimo

Ya tenemos el modelo entrenado. Ahora vamos a mirar, no solo si acierta o no, sino **qué tan bien separa** a los clientes que cancelan de los que no, usando toda la gama de probabilidades que calcula.


In [ ]:
# predict_proba() nos da la probabilidad de cada clase; nos interesa la probabilidad de "Cancela" (columna 1)
proba_test = modelo.predict_proba(X_test_s)[:, 1]

# roc_curve calcula, para muchos umbrales distintos, la tasa de falsos positivos (fpr) y verdaderos positivos (tpr)
fpr, tpr, umbrales = roc_curve(y_test, proba_test)

# auc calcula el área bajo esa curva: un solo número que resume qué tan bueno es el modelo
valor_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color="#E8820C", linewidth=2.5, label=f"Modelo (AUC = {valor_auc:.2f})")
plt.plot([0, 1], [0, 1], color="gray", linestyle="--", label="Azar")
plt.xlabel("Falsos positivos")
plt.ylabel("Verdaderos positivos")
plt.title("Curva ROC — Modelo de churn")
plt.legend()
plt.show()


In [ ]:
# INTERPRETACIÓN
print("Interpretación:")
print(f"El AUC de este modelo es {valor_auc:.2f}.")
print("Recordemos la regla práctica de la clase: 0.5 es azar puro, 1.0 es un modelo perfecto.")
print("Un AUC de 0.80 o más ya se considera útil para tomar decisiones de negocio con este modelo.")


In [ ]:
# ¿Cuál es el mejor punto de corte (umbral)? Usamos el estadístico de Youden: tpr - fpr máximo.
# Es una forma simple y estándar de encontrar el punto donde el modelo separa mejor a los dos grupos.
indice_optimo = np.argmax(tpr - fpr)
umbral_optimo = umbrales[indice_optimo]

print(f"Punto de corte óptimo sugerido: {umbral_optimo:.2f}")
print(f"En ese punto: verdaderos positivos = {tpr[indice_optimo]:.2f}, falsos positivos = {fpr[indice_optimo]:.2f}")


In [ ]:
# INTERPRETACIÓN
print("Interpretación:")
print(f"Este umbral ({umbral_optimo:.2f}) es un punto de partida estadístico, no la última palabra.")
print("Como vimos en clase, la empresa podría bajarlo si el costo de perder un cliente es muy alto,")
print("o subirlo si contactar clientes de forma innecesaria también tiene un costo importante.")


## Parte 3 · Segmentación de clientes con K-Means

Ahora usamos la probabilidad de churn que acabamos de calcular, junto con `tenure` y `MonthlyCharges`, para agrupar a los clientes en 3 segmentos de riesgo.


In [ ]:
# Calculamos la probabilidad de churn para TODOS los clientes (no solo el test), para segmentar a toda la base
proba_todos = modelo.predict_proba(scaler.transform(X))[:, 1]

# Armamos la tabla que le vamos a dar a K-Means: solo 3 variables numéricas e interpretables
X_cluster = pd.DataFrame({
    "churn_prob": proba_todos,
    "tenure": df_modelo["tenure"].values,
    "MonthlyCharges": df_modelo["MonthlyCharges"].values,
})

# Escalamos también para el clustering: K-Means es sensible a la escala de las variables
scaler_cluster = StandardScaler()
X_cluster_s = scaler_cluster.fit_transform(X_cluster)

print("Variables usadas para segmentar:", X_cluster.columns.tolist())
X_cluster.head()


In [ ]:
# Método del codo: probamos varios valores de k y guardamos la inercia de cada uno
inercias = []
valores_k = range(1, 9)

for k in valores_k:
    km_prueba = KMeans(n_clusters=k, random_state=42, n_init=10)
    km_prueba.fit(X_cluster_s)
    inercias.append(km_prueba.inertia_)

plt.figure(figsize=(6, 4))
plt.plot(list(valores_k), inercias, marker="o", color="#E8820C")
plt.xlabel("Número de clusters (k)")
plt.ylabel("Inercia")
plt.title("Método del codo")
plt.show()


In [ ]:
# INTERPRETACIÓN
print("Interpretación:")
print("Busquen en el gráfico el punto donde la línea deja de bajar fuerte y se vuelve más plana.")
print("En este dataset, ese 'codo' aparece alrededor de k = 3, así que usamos 3 segmentos,")
print("igual que en la clase: riesgo bajo, riesgo medio y riesgo alto.")


In [ ]:
# Aplicamos K-Means con 3 clusters, el número que justificamos con el método del codo
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_modelo["segmento"] = kmeans.fit_predict(X_cluster_s)
X_cluster["segmento"] = df_modelo["segmento"]

# Describimos cada segmento con el promedio de sus variables
resumen_segmentos = X_cluster.groupby("segmento").mean().round(2)
resumen_segmentos["cantidad_clientes"] = X_cluster.groupby("segmento").size()
resumen_segmentos


In [ ]:
# INTERPRETACIÓN
print("Interpretación:")
print("Miren la tabla anterior: el segmento con churn_prob más alto y tenure más bajo")
print("es el segmento de riesgo alto. El de churn_prob más bajo y tenure más alto es riesgo bajo.")
print("Anoten cuál número de segmento (0, 1 o 2) le corresponde a cada nivel de riesgo en su ejecución,")
print("porque K-Means no asigna nombres, solo números, y ese orden puede cambiar cada vez que se ejecuta.")


In [ ]:
# Visualización rápida: tenure vs. MonthlyCharges, coloreado por segmento
plt.figure(figsize=(6.5, 5))
colores = ["#3E8E5A", "#E8820C", "#C0392B"]
for seg in sorted(X_cluster["segmento"].unique()):
    subset = X_cluster[X_cluster["segmento"] == seg]
    plt.scatter(subset["tenure"], subset["MonthlyCharges"], label=f"Segmento {seg}",
                color=colores[seg % len(colores)], alpha=0.7)
plt.xlabel("Tenure (meses)")
plt.ylabel("Cargo mensual ($)")
plt.title("Segmentos de clientes por riesgo de churn")
plt.legend()
plt.show()


## Parte 4 · Su turno: describan los segmentos y propongan estrategias

Completen esta celda de Markdown (haganle doble clic para editarla) con lo que su grupo discutió en clase. Usen los números reales de la tabla `resumen_segmentos` de arriba, no valores genéricos.

**Segmento de riesgo bajo (número: ___)**
- Características: ...
- Estrategia de retención propuesta (1 acción, enfocada en fidelización): ...

**Segmento de riesgo medio (número: ___)**
- Características: ...
- Estrategias de retención propuestas (2 acciones): ...

**Segmento de riesgo alto (número: ___)**
- Características: ...
- Estrategias de retención propuestas (3 acciones, concretas y específicas): ...


## Parte 5 · Actividad de investigación (individual o en grupo)

En el código de este notebook usamos varias funciones de `pandas` y `scikit-learn`. Investiguen por su cuenta **la función principal de cada una** y expliquen, con sus propias palabras, **cómo se conecta con el tema de la semana** (riesgo, scoring, ROC/AUC o segmentación).

No copien la documentación literal: la idea es que la expliquen como se la explicarían a un compañero.

| # | Función | ¿Qué hace? (investíguenlo) | ¿Cómo se conecta con la Semana 11? |
|---|---|---|---|
| 1 | `pd.to_numeric()` | | |
| 2 | `pd.get_dummies()` | | |
| 3 | `train_test_split()` | | |
| 4 | `StandardScaler().fit_transform()` | | |
| 5 | `LogisticRegression().fit()` | | |
| 6 | `predict_proba()` | | |
| 7 | `classification_report()` | | |
| 8 | `roc_curve()` | | |
| 9 | `auc()` | | |
| 10 | `KMeans().fit_predict()` | | |
| 11 | `.inertia_` (atributo de KMeans) | | |
| 12 | `.groupby().mean()` | | |

**Instrucciones de entrega:** completen la tabla directamente en esta celda de Markdown (o en el documento que su profesor indique en Blackboard) y guarden el notebook. Esta actividad no tiene nota esta semana, pero es la base para el Producto de Investigación y para el crucigrama de repaso que vamos a revisar en clase.


## Cierre

Con esto ya tienen: el modelo de churn reentrenado, la lectura de la curva ROC y el punto de corte óptimo, y los 3 segmentos de clientes con sus estrategias de retención.

La próxima semana (Módulo 12) vamos a usar estas mismas herramientas, pero para identificar oportunidades de negocio en lugar de solo reducir riesgos: validación cruzada, `GridSearchCV` y un primer acercamiento a interpretabilidad con SHAP.
